In [ ]:
# Imports
from typing import Literal, List, Union, Optional, Annotated, Tuple, Callable
from collections import defaultdict
import time
import re

# Dependencies
import requests
import pandas as pd
import mwparserfromhell
from pydantic import BaseModel, Field

# Library
from classes import pipeline

# Definition
JSONType = dict[str]

In [ ]:
# Params
urls = []
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
first_param = {
	"action": "parse",
	"page": "List of Cardfight!! Vanguard Booster Sets",
	"format": "json"
}
rules = [
    (r"^DZ", "DZ"),
    (r"^D", "D"),
    (r"^G", "G"),
    (r"^V", "V"),
    (r"^Booster", "LB")
]

In [ ]:
# Url Parser
web = pipeline.MediaWikiAPI()
pipeline = pipeline.VanguardPipeline(
	pipeline.VanguardScrapper(web),
	pipeline.VanguardParser(),
	pipeline.VanguardClassifier(rules),
	pipeline.VanguardStorage()
)
curl = pipeline.scrapper.api.get(params=first_param, headers=header)
crude_data = pipeline.scrapper.obtain_links(curl)
no_main_sets = pipeline.parser.separate_boosters(crude_data)
pipeline.parser.remove_from_list(no_main_sets, [
	"Lyrical Monasterio", "The Mask Collection"
])
pipeline.parser.remove_from_list(crude_data, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])


In [ ]:
crude_data

In [ ]:
for i in crude_data:
	key = pipeline.classifier.classify(i)
	pipeline.storage._add_item(key, i)

In [ ]:
pipeline.storage.dz

In [ ]:
attr = ["LB", "G"]
for at in attr:
	lst = getattr(pipeline.storage, at.lower())
	lst.sort(key=pipeline.classifier.obtain_set_number)
del attr
del lst

In [ ]:
consults = pipeline.parser.make_consults("consult", pipeline.storage.lb)

In [ ]:
consults[0]

In [ ]:
pipeline.storage.dz

In [ ]:
pipeline.storage.ll